# 04 · Genetic algorithm and strategy comparison

Milestone 4. Engine 3 answers **Q4** (which combination of customers to target) and **Q5** (how much more profit than the existing strategies).

```
maximise   sum_i x_i * EP_i                       x_i in {0, 1}, eligible customers only
subject to sum_i x_i * C_i            <= budget                (£50,000)
           sum_i x_i                  <= max contacts          (6,000)
           sum_{i in g} x_i           <= channel capacity      (calls 4,500, digital 2,500)
           share of each segment      <= 60%
           share Growing >= 15%, share Dormant >= 10%
           share of protected 75+     <= 2%
```

*All economic figures are illustrative assumptions, not bank performance.*

In [1]:
from dataclasses import asdict, replace
import time
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pacopt import config as cfg, optimize as op, pipeline as pl, charts as ch

pio.renderers.default = 'notebook_connected'
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
t = pl.load_campaign_table()
pr = op.build_problem(t)
print(f'{len(t):,} customers | {pr.n:,} eligible enter the optimisation | {(pr.ep > 0).sum():,} with positive EP')
pd.DataFrame({'constraints': pd.Series(asdict(cfg.CONSTRAINTS)), }).astype(str)

10,000 customers | 9,041 eligible enter the optimisation | 6,700 with positive EP


,constraints
budget,50000.0
max_contacts,6000
channel_capacity,"{'calls': 4500, 'digital': 2500}"
segment_max_share,"{'High value': 0.6, 'Growing': 0.6, 'Dormant':..."
segment_min_share,"{'Growing': 0.15, 'Dormant': 0.1}"
protected_max_share,0.02


## 1. Design (sub-tasks 4.1-4.4)
| Component | Choice |
|---|---|
| Chromosome | Binary NumPy vector, one gene per eligible customer with positive EP, plus the members of segments with a minimum share (a negative-EP customer can only help by meeting a minimum). The population is a (100 x n) boolean matrix, so every operator is vectorised. |
| Fitness | Total EP of the list, minus a large penalty per unit of constraint violation (the fallback if repair ever leaves a violation). |
| Repair | Shared constraint engine: drop the weakest selections until channel, contact and budget limits hold; trim over-represented segments; add the best members of under-represented ones; then **fill** spare capacity in EP-per-£ order (D-036). |
| Selection / crossover / mutation | Tournament (k = 3) / uniform (p = 0.9) / asymmetric bit-flip: about 30 genes switched on and 0.5 off per child, so with repair a mutation works as a set of swaps. |
| Elitism / stopping | Top 2% kept / 300 generations or 50 without improvement. |
| Seeding | 4 of 100 chromosomes start from the Greedy list (and perturbed copies); the rest are random sparse lists at the density the budget allows. An unseeded run is reported too. |

Every strategy, not just the GA, goes through the same constraint engine. `check_constraints()` then audits every list independently, from the campaign table and the customer IDs alone.

## 2. Run all strategies (sub-task 4.5)

In [2]:
t0 = time.perf_counter()
comp = op.compare_strategies(t)
print(f'All strategies in {time.perf_counter() - t0:.0f}s | GA: {comp.ga.generations} generations, '
      f'{comp.ga.runtime_s:.1f}s | MILP: {comp.milp.message.split("(")[0].strip()} in {comp.milp.runtime_s:.0f}s, '
      f'gap {comp.milp.mip_gap:.1e}')
cols = ['contacts', 'contacts_calls', 'contacts_digital', 'spend', 'expected_conversions', 'expected_contribution',
        'roi', 'incremental_vs_random', 'uplift_vs_random', 'pct_of_optimum', 'feasible']
summary = comp.table[cols]
summary.style.format({'spend': '£{:,.0f}', 'expected_conversions': '{:,.0f}', 'expected_contribution': '£{:,.0f}',
                      'roi': '{:.2f}', 'incremental_vs_random': '£{:+,.0f}', 'uplift_vs_random': '{:+.0%}',
                      'pct_of_optimum': '{:.2%}', 'contacts': '{:,}', 'contacts_calls': '{:,.0f}',
                      'contacts_digital': '{:,.0f}'})

All strategies in 44s | GA: 90 generations, 12.7s | MILP: Optimization terminated successfully. in 29s, gap 0.0e+00


,contacts,contacts_calls,contacts_digital,spend,expected_conversions,expected_contribution,roi,incremental_vs_random,uplift_vs_random,pct_of_optimum,feasible
strategy,,,,,,,,,,,
Random,"5,517","3,969","1,548","£50,000",676,"£60,660",1.21,£+0,+0%,58.44%,True
Propensity,"4,275","4,203",72,"£49,992",839,"£91,244",1.83,"£+30,584",+50%,87.91%,True
Greedy,"6,000","3,500","2,500","£46,364",884,"£103,752",2.24,"£+43,092",+71%,99.96%,True
Genetic Algorithm,"6,000","3,600","2,400","£47,358",891,"£103,793",2.19,"£+43,133",+71%,100.00%,True
Rule-based,"5,374","4,002","1,372","£49,998",756,"£83,819",1.68,"£+23,159",+38%,80.76%,True
MILP optimum,"6,000","3,600","2,400","£47,358",891,"£103,793",2.19,"£+43,133",+71%,100.00%,True


*ROI = expected contribution / spend. Random is the mean of 50 seeds (standard deviation shown below). Rule-based = highest balance first.*

In [3]:
r = comp.random_runs['expected_contribution']
print(f'Random baseline over {len(r)} seeds: mean £{r.mean():,.0f}, std £{r.std():,.0f}, '
      f'range £{r.min():,.0f} to £{r.max():,.0f}; all feasible: {comp.random_runs.feasible.all()}')

Random baseline over 50 seeds: mean £60,660, std £1,017, range £58,166 to £62,590; all feasible: True


## 3. Expected contribution by strategy (the Screen 3 view, sub-task 4.7)

In [4]:
names = list(op.CHART_STRATEGIES)
vals = comp.table.loc[names, 'expected_contribution']
fig = go.Figure(go.Bar(x=names, y=vals, marker_color=ch.BAR_BLUE, marker_cornerradius=8,
                       text=[f'£{v / 1e3:,.1f}K' for v in vals], textposition='outside',
                       textfont=dict(color=ch.INK_2),
                       hovertemplate='%{x}<br>£%{y:,.0f}<extra></extra>'))
ch.style(fig, 'Expected contribution by strategy (£50K budget)', height=430, showlegend=False)
ch.reference_line(fig, -0.5, comp.milp.expected_profit, len(names) - 0.5, comp.milp.expected_profit,
                  label=f'Theoretical max (MILP) £{comp.milp.expected_profit / 1e3:,.1f}K', label_at='start')
fig.update_yaxes(tickprefix='£', tickformat='~s', rangemode='tozero', range=[0, comp.milp.expected_profit * 1.15])
fig.update_layout(bargap=0.35)
fig.show()

In [5]:
best_baseline = comp.table.loc[['Random', 'Propensity', 'Rule-based'], 'expected_contribution']
ga = comp.table.loc['Genetic Algorithm']
print('Q5: incremental expected contribution of the GA list')
for k, v in best_baseline.items():
    print(f'  vs {k:<11s} £{ga.expected_contribution - v:>+10,.0f}  ({ga.expected_contribution / v - 1:+.0%})')
print(f"  vs Greedy      £{ga.expected_contribution - comp.table.loc['Greedy', 'expected_contribution']:>+10,.0f}")
print(f'GA reaches {ga.pct_of_optimum:.2%} of the MILP optimum.')

Q5: incremental expected contribution of the GA list
  vs Random      £   +43,133  (+71%)
  vs Propensity  £   +12,549  (+14%)
  vs Rule-based  £   +19,974  (+24%)
  vs Greedy      £       +41
GA reaches 100.00% of the MILP optimum.


**Reading the result honestly.** The GA beats Random by about 70% and Propensity by about 14%. Most of that gain comes from **ranking by profit per £, not from the GA itself**: Greedy (EP / C order) already reaches 99.96% of the optimum. In the default scenario the only limit binding at the optimum is the 6,000-contact cap. The optimum leaves about £2.6K of budget and 100 digital slots unused, and no segment share binds (see section 6). So the problem is close to a knapsack, which a greedy ratio rule solves almost exactly. Greedy fills all 2,500 digital slots first because they are cheap. Once the head-count cap binds, though, a contact's EP matters more than its EP per £. The GA finds the last £41 by trading 100 digital contacts for 100 calls with higher EP, a swap a single ranking cannot see. Section 7 shows scenarios where the constraints interact and the GA's gain over Greedy is material.

## 4. Convergence (sub-task 4.6)
The seeded run starts from the Greedy list and is already near the optimum at generation 0, so the unseeded run is the clearer view of the algorithm's search.

In [6]:
ga0 = op.run_ga(pr, replace(cfg.GA, greedy_seeds=0))
print(f'Unseeded GA: £{ga0.expected_profit:,.0f} ({ga0.expected_profit / comp.milp.expected_profit:.2%} of optimum) '
      f'in {ga0.generations} generations, {ga0.runtime_s:.1f}s | seeded: {comp.ga.generations} generations, {comp.ga.runtime_s:.1f}s')
fig = go.Figure()
for h, name, color in ((ga0.history, 'Unseeded', ch.BLUE), (comp.ga.history, 'Greedy-seeded', ch.ORANGE)):
    fig.add_scatter(x=h['generation'], y=h['best'], name=f'{name}: best', mode='lines', line=dict(color=color, width=2),
                    hovertemplate=name + ' best<br>generation %{x}<br>£%{y:,.0f}<extra></extra>')
    fig.add_scatter(x=h['generation'], y=h['mean'], name=f'{name}: population mean', mode='lines',
                    line=dict(color=color, width=1.5, dash='dot'),
                    hovertemplate=name + ' mean<br>generation %{x}<br>£%{y:,.0f}<extra></extra>')
g_end = max(ga0.generations, comp.ga.generations)
ch.reference_line(fig, 0, comp.milp.expected_profit, g_end, comp.milp.expected_profit, 'MILP optimum')
greedy_ep = comp.table.loc['Greedy', 'expected_contribution']
ch.style(fig, 'GA convergence: best and mean fitness per generation', height=450)
fig.update_xaxes(title='Generation')
fig.update_yaxes(title='Expected contribution (£)', tickprefix='£', tickformat=',.0f',
                 range=[ga0.history['mean'].min() * 0.995, comp.milp.expected_profit * 1.003])
fig.show()
print(f'Greedy for reference: £{greedy_ep:,.0f}')

Unseeded GA: £103,793 (100.00% of optimum) in 107 generations, 15.9s | seeded: 90 generations, 12.7s


Greedy for reference: £103,752


## 5. The GA target list: channel and segment mix (Q4)

In [7]:
seg_mix = op.selection_mix(t, comp, 'segment')
ch_mix = op.selection_mix(t, comp, 'channel_group')
display(seg_mix.style.format('{:.1%}').set_caption('Segment share of each list (Random = seed 0)'))
ch_mix.style.format('{:.1%}').set_caption('Channel share of each list')

,Random,Propensity,Greedy,Rule-based,Genetic Algorithm,MILP optimum
segment,,,,,,
High value,27.5%,56.4%,38.6%,36.8%,39.1%,39.1%
Growing,15.7%,33.0%,17.0%,17.4%,17.7%,17.7%
Dormant,27.2%,10.7%,25.9%,45.9%,26.1%,26.1%
Low engagement,29.7%,0.0%,18.6%,0.0%,17.1%,17.1%


,Random,Propensity,Greedy,Rule-based,Genetic Algorithm,MILP optimum
channel_group,,,,,,
calls,72.1%,98.3%,58.3%,74.5%,60.0%,60.0%
digital,27.9%,1.7%,41.7%,25.5%,40.0%,40.0%


In [8]:
tl = op.target_list(t, comp)
print(f'GA target list: {len(tl):,} customers, spend £{tl.contact_cost.sum():,.0f}, '
      f'expected conversions {tl.propensity.sum():,.0f}, expected contribution £{tl.expected_profit.sum():,.0f}')
tl.head(10).round(3)

GA target list: 6,000 customers, spend £47,358, expected conversions 891, expected contribution £103,793


,customer_id,segment,channel,propensity,value,contact_cost,break_even_cost,expected_profit,ep_per_cost,age,flag_age_75plus
0,C141694,High value,mobile_call,0.382,1375.790,12.0,525.287,513.287,42.774,60,0
1,C144522,High value,mobile_call,0.696,614.680,12.0,427.761,415.761,34.647,42,0
2,C143291,High value,mobile_call,0.527,643.450,12.0,338.829,326.829,27.236,77,1
3,C141383,High value,mobile_call,0.667,444.370,12.0,296.227,284.227,23.686,48,0
4,C131223,High value,mobile_call,0.414,701.988,12.0,290.276,278.276,23.190,61,0
5,C142718,High value,mobile_call,0.742,367.055,12.0,272.207,260.207,21.684,61,0
6,C140555,High value,mobile_call,0.880,306.505,12.0,269.724,257.724,21.477,26,0
7,C126228,High value,landline_call,0.150,1705.000,10.0,256.209,246.209,24.621,59,0
8,C144679,High value,mobile_call,0.704,357.150,12.0,251.495,239.495,19.958,60,0
9,C137608,High value,mobile_call,0.639,369.943,12.0,236.388,224.388,18.699,32,0


## 6. Constraint compliance (sub-task 4.3)
`check_constraints()` audits each list from the campaign table and customer IDs alone. Below: the value of every constraint for each strategy, with ✓ when it holds.

In [9]:
audit = {}
for name in comp.selections:
    ck = op.check_constraints(t, comp.selected_ids(name)).set_index('constraint')
    audit[name] = [(f'{v:.1%}' if 'share' in c.lower() else f'{v:,.0f}') + (' ✓' if ok else ' ✗')
                   for c, v, ok in zip(ck.index, ck['value'], ck['ok'])]
limits = op.check_constraints(t, comp.selected_ids('Greedy')).set_index('constraint')['limit']
audit = pd.DataFrame(audit, index=limits.index)
audit.insert(0, 'limit', [f'{v:.0%}' if 'share' in c.lower() else f'{v:,.0f}' for c, v in limits.items()])
audit

,limit,Random,Propensity,Greedy,Rule-based,Genetic Algorithm,MILP optimum
constraint,,,,,,,
Unique customers,0,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓
Known customers,0,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓
Eligible customers only,0,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓,0 ✓
Budget (£),"50,000","50,000 ✓","49,992 ✓","46,364 ✓","49,998 ✓","47,358 ✓","47,358 ✓"
Contacts,"6,000","5,508 ✓","4,275 ✓","6,000 ✓","5,374 ✓","6,000 ✓","6,000 ✓"
Channel capacity: calls,"4,500","3,972 ✓","4,203 ✓","3,500 ✓","4,002 ✓","3,600 ✓","3,600 ✓"
Channel capacity: digital,"2,500","1,536 ✓",72 ✓,"2,500 ✓","1,372 ✓","2,400 ✓","2,400 ✓"
Max share: High value,60%,27.5% ✓,56.4% ✓,38.6% ✓,36.8% ✓,39.1% ✓,39.1% ✓
Max share: Growing,60%,15.7% ✓,33.0% ✓,17.0% ✓,17.4% ✓,17.7% ✓,17.7% ✓


In [10]:
bind = op.check_constraints(t, comp.selected_ids('MILP optimum')).set_index('constraint')
slack = (bind['limit'] - bind['value']).where(~bind.index.str.startswith('Min'), bind['value'] - bind['limit'])
print('Binding constraints at the optimum (slack 0):', ', '.join(slack.index[(slack.abs() < 1e-9) & ~slack.index.str.contains('customers')]))

Binding constraints at the optimum (slack 0): Contacts


## 7. Where the GA matters: stressed constraint scenarios
Same customers and economics; only the campaign limits change. These are what-ifs, not new defaults (D-037).

In [11]:
scenarios = {
    'Default': cfg.CONSTRAINTS,
    'Few call agents (calls 1,500; Growing >= 20%, Dormant >= 15%)':
        replace(cfg.CONSTRAINTS, channel_capacity={'calls': 1_500, 'digital': 2_500},
                segment_min_share={'Growing': 0.20, 'Dormant': 0.15}),
    'Balanced mix (High value <= 30%; Growing, Dormant >= 25%)':
        replace(cfg.CONSTRAINTS, segment_max_share={**{s: 0.60 for s in cfg.SEGMENTS}, 'High value': 0.30},
                segment_min_share={'Growing': 0.25, 'Dormant': 0.25}),
}
rows = {}
for name, cc in scenarios.items():
    c = comp if name == 'Default' else op.compare_strategies(t, cc)
    tb = c.table['expected_contribution']
    rows[name] = {**tb.to_dict(), 'GA gain vs Greedy': tb['Genetic Algorithm'] - tb['Greedy'],
                  'Greedy % of optimum': tb['Greedy'] / tb['MILP optimum'],
                  'GA % of optimum': tb['Genetic Algorithm'] / tb['MILP optimum'],
                  'GA runtime (s)': c.ga.runtime_s, 'all feasible': bool(c.table['feasible'].all())}
stress = pd.DataFrame(rows).T
money = list(op.STRATEGY_ORDER) + ['GA gain vs Greedy']
stress.style.format({**{k: '£{:,.0f}' for k in money}, 'Greedy % of optimum': '{:.2%}',
                     'GA % of optimum': '{:.2%}', 'GA runtime (s)': '{:.0f}'})

,Random,Propensity,Greedy,Genetic Algorithm,Rule-based,MILP optimum,GA gain vs Greedy,Greedy % of optimum,GA % of optimum,GA runtime (s),all feasible
Default,"£60,660","£91,244","£103,752","£103,793","£83,819","£103,793",£41,99.96%,100.00%,13,True
"Few call agents (calls 1,500; Growing >= 20%, Dormant >= 15%)","£31,221","£77,161","£78,082","£81,588","£42,500","£82,939","£3,507",94.14%,98.37%,36,True
"Balanced mix (High value <= 30%; Growing, Dormant >= 25%)","£58,373","£96,484","£98,239","£99,809","£80,492","£99,939","£1,570",98.30%,99.87%,95,True


When a segment minimum or a tight channel cap binds, the ratio order makes locally good choices that clash with the other limits, and repairing them afterwards costs profit. The GA searches across whole lists, so it recovers most of that loss. This is the case the plan anticipated: the GA's value rises with how much the constraints interact.

The stressed scenarios take longer, because repair works harder when several share limits bind at once. The 60-second target applies to the default run. For dashboard re-runs, `GAConfig.time_limit_s` caps the wall-clock time, and the best feasible list found so far is returned.

## 8. Save the optimisation artifacts

In [12]:
pl.save_optimisation(t, comp)
for p in (pl.STRATEGY_PATH, pl.TARGET_LIST_PATH, pl.GA_HISTORY_PATH, pl.SELECTIONS_PATH):
    print(p.relative_to(cfg.PROJECT_ROOT))

artifacts\strategy_comparison.csv
artifacts\ga_target_list.csv
artifacts\ga_history.csv
artifacts\strategy_selections.parquet


**Milestone 4 output:** the strategy comparison, the GA target list, the convergence history and every strategy's selection, all under `artifacts/`. The dashboard (M5) reads these and re-runs the optimiser when the sliders change.